# EXAONE-3.5-7.8B LoRA 파인튜닝 (RunPod GPU)

`data/final/train.jsonl`(2,808) + `valid.jsonl`(311) 로 EXAONE 플래너를 LoRA 파인튜닝한다.
학습 로직은 `sft_pipeline/train/train_lora.py` 가 전부 담당한다 — 이 노트북은 RunPod GPU에서
환경 셋업 → 데이터 확인 → 학습 실행 → 스모크 점검만 한다.

**전제**: RunPod *PyTorch* 템플릿(CUDA 12.x, A100/H100 권장)에서 이 노트북을 연다.
리포지토리는 `/workspace/mongle-ai` 에 clone 되어 있다고 가정한다.
데이터는 셀 `1.5` 가 S3(`$SFT_BUCKET/$SFT_PREFIX`)에서 `data/final/` 로 내려받는다.
(로컬에서 미리 `aws s3 cp` 로 train/valid.jsonl 을 올려둘 것.)

In [ ]:
# 0. GPU 확인 (없으면 학습 불가)
!nvidia-smi

In [ ]:
# 1. 리포 위치로 이동 (RunPod 볼륨 경로에 맞게 수정)
%cd /workspace/mongle-ai
!pwd && ls sft_pipeline/data/final/

In [ ]:
# 1.5 데이터를 S3에서 받기 (data/final 이 비어있을 때만 필요)
#     로컬에서 미리 올려둔다:
#       aws s3 cp sft_pipeline/data/final/train.jsonl s3://$BUCKET/$PREFIX/train.jsonl
#       aws s3 cp sft_pipeline/data/final/valid.jsonl s3://$BUCKET/$PREFIX/valid.jsonl
import os

BUCKET = os.environ.get("SFT_BUCKET", "mongle-village-prod-962214557220-ap-northeast-2-an")
PREFIX = os.environ.get("SFT_PREFIX", "mongle-village/sft/datasets/planner")

!mkdir -p sft_pipeline/data/final
!aws s3 cp s3://{BUCKET}/{PREFIX}/train.jsonl sft_pipeline/data/final/train.jsonl
!aws s3 cp s3://{BUCKET}/{PREFIX}/valid.jsonl sft_pipeline/data/final/valid.jsonl
!ls -lh sft_pipeline/data/final/

In [ ]:
# 2. 의존성 설치 (train_lora.py docstring 기준)
#    unsloth 는 trl/transformers 보다 먼저 import 되어야 하므로 함께 설치한다.
!pip install -q "unsloth[colab-new]" trl peft accelerate bitsandbytes datasets

In [ ]:
# 3. 데이터 정합성 확인 — 줄 수 + JSON 파싱 + messages 키 + kind 분포
import json
from pathlib import Path

for name in ["train", "valid"]:
    path = Path(f"sft_pipeline/data/final/{name}.jsonl")
    rows = [json.loads(l) for l in path.open() if l.strip()]
    assert all("messages" in r for r in rows), f"{name}: messages 키 누락"
    kinds = {}
    for r in rows:
        try:
            k = json.loads(r["messages"][-1]["content"]).get("kind", "?")
        except Exception:
            k = "non_json"
        kinds[k] = kinds.get(k, 0) + 1
    print(f"{name}: {len(rows)}개 | kind 분포: {kinds}")

In [ ]:
# 4. LoRA 학습 실행 — train_lora.py 가 모델 로딩·마스킹·EOS·저장을 모두 처리.
#    --model 기본값이 이미 EXAONE-3.5-7.8B-Instruct 라 생략해도 된다.
#    GPU 메모리에 따라 --batch / --grad-accum / --max-seq-len 조정.
!python -m sft_pipeline.train.train_lora \
    --train sft_pipeline/data/final/train.jsonl \
    --valid sft_pipeline/data/final/valid.jsonl \
    --out outputs/exaone-planner-lora \
    --epochs 2 \
    --batch 2 \
    --grad-accum 4 \
    --lora-r 16 \
    --lora-alpha 16

In [ ]:
# 5. 어댑터 저장 확인
!ls -lh outputs/exaone-planner-lora/

In [ ]:
# 6. 스모크 테스트 — 학습한 어댑터로 1건 생성해 JSON 파싱되는지 확인.
#    (train_lora.py 와 동일하게 unsloth 를 먼저 import)
from unsloth import FastLanguageModel
import json

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="outputs/exaone-planner-lora",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# valid 첫 샘플의 user 발화로 생성 (assistant turn 제외)
sample = json.loads(open("sft_pipeline/data/final/valid.jsonl").readline())
prompt_msgs = [m for m in sample["messages"] if m["role"] != "assistant"]
inputs = tokenizer.apply_chat_template(
    prompt_msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=1024, temperature=0.0)
text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print(text)
print("\n--- JSON 파싱 ---")
print("OK" if json.loads(text) else "FAIL")

## 다음 단계

- validation loss 가 0.2 미만이면 과적합 경고 → epochs 줄이거나 데이터 다양화.
- 어댑터를 S3/HF 로 업로드 후 서빙(RunPod LLM 워커)의 LoRA repo 로 배선.
- 정량 평가는 `sft_pipeline/eval/planner_loop_eval.ipynb` 참고.